In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from design import Design
from geometry import create_barrier

project_name = "SynRM_test"
design_name = "Design01"
path_data = os.path.join(os.getcwd(), 'data')
path_results = 'results'
for path in [path_data, path_results]:
    os.makedirs(path, exist_ok=True)
file_name_aedt = f'{path_data}/{project_name}.aedt'
save_design = False
plot_design = True
n_designs = 3

# Define constants
AEDT_VERSION = "2024.2"
NUM_CORES = 4
NG_MODE = True  #non-graphical mode
CLS_EXIT = True #close on exit

if not os.path.exists(file_name_aedt):
    design = Design.create(
        project_name, design_name, file_name_aedt,
        version=AEDT_VERSION,
        non_graphical=NG_MODE,
        new_desktop=False,
        close_on_exit=CLS_EXIT,
    )
else:
    design = Design.load(
        file_name_aedt,
        version=AEDT_VERSION,
        non_graphical=NG_MODE,
        new_desktop=False,
        close_on_exit=CLS_EXIT,
    )

In [ ]:
w_mins_base = np.array([3, 2.5, 2.5, 2]) - 1.0

# Loop over the number of designs
for i in range(0, n_designs):
    # Parameterize the design
    w_mins = w_mins_base + 0.5*np.random.random(4)
    w_mids = w_mins
    y_min0 = 13+4*np.random.random()
    y_min1 = y_min0+w_mins[0]+2+3*np.random.random()
    y_min2 = y_min1+w_mins[1]+2+3*np.random.random()
    y_min3 = y_min2+w_mins[2]+2+2*np.random.random()
    y_mins = np.array([y_min0, y_min1, y_min2, y_min3])
    y_mids = y_mins + np.array([2, 1.5, 1, 0.5])
    thetas = [1, 8, 15, 21]
    w_maxs = w_mins - np.array([0.5, 0.5, 0.5, 0])
    
    # Add rotor
    design.add_rotor()

    # Add barriers
    for args in zip(y_mins, w_mins, y_mids, w_mids, thetas, w_maxs):
        x_all, y_all = create_barrier(design, *args)
        xy_all = np.vstack((x_all, y_all)).T
        design.add_rotor_barriers(xy_all)

    # Compute the torque
    Tor = design.compute(NUM_CORES)
    TorAvg, TorRmsAC, TorRippleRms = design.analyze_results(Tor)

    # Potentially save the design
    file_design = f'{path_results}/adesign_{i}'
    if save_design:
        design.save_design(file_design)

    # Delete the rotor
    design.delete_rotor()

    # Potentially plot the desing
    if plot_design:
        plt.figure()
        for args in zip(y_mins, w_mins, y_mids, w_mids, thetas, w_maxs):
            x_all, y_all = create_barrier(design, *args)
            plt.plot(x_all, y_all)
        alphas = np.linspace(0*np.pi,2*np.pi/4,100)
        for r in [design.rotor_r_min, design.rotor_r_max]:
            plt.plot(r*np.cos(alphas), r*np.sin(alphas))
        plt.title(f'Torque mean value: {np.round(TorAvg,2)} Nm, ripple relative value: {np.round(TorRippleRms,2)} %')
        plt.savefig(f'{file_design}.png')
        plt.close()

    # Save the points
    np.savez(f'{file_design}.npz',
         y_mins=y_mins,
         w_mins=w_mins,
         y_mids=y_mids,
         w_mids=w_mids,
         thetas=thetas,
         w_maxs=w_maxs)

In [ ]:
design.close_project()